# D1 — Frozen AraBERT-twitter: `[CLS]` vs. mean pooling → logistic regression

`aubmindlab/bert-base-arabertv02-twitter`, encoder frozen. Each tweet → `[CLS]` vector and attention-mask-weighted mean of the tokens →
StandardScaler → LogisticRegression (`class_weight="balanced"`, `C` picked on val) → threshold tuned on val (`fit_with_threshold`).
Fit on train only; test stays sealed.

In [ ]:
import os
import sys

REPO = "Arabic-sarcasm"

if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/cad0m/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull

%cd /content/{REPO}

if f"/content/{REPO}" not in sys.path:
    sys.path.insert(0, f"/content/{REPO}")

!pip install -q uv
!uv pip install --system emoji tashaphyne

In [ ]:
# arabert pins an old emoji (1.x, no Arabic names): install it without deps, then force emoji>=2.16.
# If you see a KeyError in emoji.demojize, restart the runtime once and re-run from the top.
%pip install -q --no-deps arabert
%pip install -q -U "emoji>=2.16" pyarabic sentencepiece protobuf

In [ ]:
import time

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoTokenizer

from src.data import get_data
from src.evaluate import log_result, metrics, plot_results, save_predictions, show
from src.preprocess import build_pipeline
from src.train import fit_with_threshold, set_seed

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

In [ ]:
train, val, test, report = get_data()
y_train, y_val = train["sarcasm"].values, val["sarcasm"].values
print(f"train: {len(train)} | val: {len(val)}")

In [ ]:
BASE = ["html", "mentions", "hashtags"]
EMOJI = ["emoji"]
NORM = ["diacritics", "kashida", "letters", "elongation"]
LIGHT = ["light_stem"]
URLS = ["urls"]

COMBOS = {
    # same names / definitions as 02_baseline (so rows are comparable in results.csv)
    "base":                      BASE,
    "emoji":                     BASE + EMOJI,
    "norm":                      BASE + NORM,
    "light_stem":                BASE + LIGHT,
    "emoji + norm":              BASE + EMOJI + NORM,
    "emoji + light_stem":        BASE + EMOJI + LIGHT,
    "norm + light_stem":         BASE + NORM + LIGHT,
    "emoji + norm + light_stem": BASE + EMOJI + NORM + LIGHT,
    "urls kept":                 BASE,
    # added for PLMs: URLs -> <URL>, no stemming, AraBERT's own preprocessor
    "urls replaced":             BASE + URLS,
    "emoji + urls replaced":     BASE + URLS + EMOJI,
    "minimal":                   ["html", "urls", "mentions"],   # what 05_finetune uses
    "arabert":                   None,
}

In [ ]:
def make_prep(steps):
    if steps is None:
        from arabert.preprocess import ArabertPreprocessor
        return ArabertPreprocessor(model_name="aubmindlab/bert-base-arabertv02-twitter").preprocess
    return build_pipeline(steps)


def prepare(frame, steps):
    prep = make_prep(steps)
    texts = [prep(tweet) for tweet in frame["tweet"]]
    return [t if t.strip() else raw for t, raw in zip(texts, frame["tweet"])]


try:
    make_prep(None)("هذا مثال 😂 @user #وسم_تجريبي http://t.co/x")   # fails here, not mid-run, if arabert is incompatible
except Exception as error:
    print("arabert preprocessing skipped:", error)
    COMBOS.pop("arabert")

print(list(COMBOS))
print("base       :", prepare(val.head(4), COMBOS["base"])[3][:100])
print("urls kept == base (BASE has no urls step), so those two rows are expected to match")

In [ ]:
def slug(name):
    return name.replace(" + ", "+").replace(" ", "_")

NAME = "aubmindlab/bert-base-arabertv02-twitter"
MAX_LEN, BATCH = 96, 64

tok = AutoTokenizer.from_pretrained(NAME)
encoder = AutoModel.from_pretrained(NAME).to(DEVICE).eval()
for p in encoder.parameters():
    p.requires_grad = False
HIDDEN = encoder.config.hidden_size
print("frozen parameters:", sum(p.numel() for p in encoder.parameters()))

In [ ]:
@torch.no_grad()
def embed(texts):
    """[CLS] and mask-weighted mean of the last layer, in the original order."""
    order = np.argsort([len(t) for t in texts])
    cls = np.zeros((len(texts), HIDDEN), dtype=np.float32)
    mean = np.zeros((len(texts), HIDDEN), dtype=np.float32)
    for i in range(0, len(texts), BATCH):
        idx = order[i:i + BATCH]
        enc = tok([texts[j] for j in idx], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            hidden = encoder(**enc).last_hidden_state.float()
        mask = enc["attention_mask"].unsqueeze(-1).float()
        cls[idx] = hidden[:, 0].cpu().numpy()
        mean[idx] = ((hidden * mask).sum(1) / mask.sum(1).clamp(min=1)).cpu().numpy()
    return cls, mean


def fit_lr(X_train, X_val):
    best = None
    for C in (0.01, 0.1, 1.0):
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(C=C, class_weight="balanced", max_iter=2000)
        )
        threshold, seconds = fit_with_threshold(model, X_train, y_train, X_val, y_val)
        proba = model.predict_proba(X_val)[:, 1]
        scores = metrics(y_val, proba, threshold)
        if best is None or scores["f1"] > best["scores"]["f1"]:
            best = dict(C=C, model=model, threshold=threshold, seconds=seconds, scores=scores, proba=proba)
    return best

In [ ]:
def run(combos):
    rows, seen = [], {}

    for name, steps in combos.items():
        train_texts, val_texts = prepare(train, steps), prepare(val, steps)
        h = hash((tuple(train_texts), tuple(val_texts)))

        if h not in seen:                       # identical inputs (base / urls kept) are computed once
            tr, va = embed(train_texts), embed(val_texts)
            seen[h] = {p: fit_lr(tr[k], va[k]) for k, p in enumerate(("cls", "mean"))}

        for pooling, best in seen[h].items():
            model_name = f"arabert_frozen_{pooling}_lr"

            log_result(
                model=model_name,
                preprocessing=name,
                seed=42,
                threshold=best["threshold"],
                train_time=best["seconds"],
                trained_params=best["model"][-1].coef_.size + 1,
                **{f"val_{k}": v for k, v in best["scores"].items()}
            )
            save_predictions(f"{model_name}_{slug(name)}", 42, val["id"], y_val, best["proba"], best["threshold"])

            show(f"{pooling} C={best['C']} | {name}", best["scores"], best["threshold"], best["seconds"])
            rows.append({"pooling": pooling, "preprocessing": name, "C": best["C"],
                         "threshold": best["threshold"], **best["scores"]})

    return pd.DataFrame(rows)

table = run(COMBOS)

In [ ]:
table.pivot(index="preprocessing", columns="pooling", values="f1").round(4)

In [ ]:
summary = table.assign(name=table["pooling"] + " | " + table["preprocessing"]).set_index("name")
plot_results(summary.nlargest(15, "f1"))